# 01 - Curated Crawl

This notebook demonstrates Stage 1: turning a documentation sitemap into a curated crawl scope. The local cells use sample URLs and the same classifier as `scripts/sitemap_to_inventory.py`.


In [ ]:
from pathlib import Path
import json
import sys
from collections import Counter, defaultdict

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'scripts').exists():
    for parent in Path.cwd().parents:
        if (parent / 'scripts').exists() and (parent / 'pyproject.toml').exists():
            REPO_ROOT = parent
            break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.sitemap_to_inventory import classify

sample_urls = [
    'https://docs.example.com/platform/widget-a/latest/index.html',
    'https://docs.example.com/platform/widget-a/latest/install.html',
    'https://docs.example.com/platform/widget-a/1.5.0/install.html',
    'https://docs.example.com/platform/widget-b/2.0.0/index.html',
    'https://docs.example.com/platform/widget-b/2.1.0/index.html',
    'https://docs.example.com/platform/widget-c/stable/ops/deploy.html',
]

rows = []
for url in sample_urls:
    source_area, version, page = classify(url, '/platform/')
    rows.append({'url': url, 'source_area': source_area, 'version': version, 'page': page})

for row in rows:
    print(row)


A source area should have one active prefix in the training corpus. Prefer `latest` when it is complete, otherwise pin a concrete version.


In [ ]:
versions_by_source_area = defaultdict(Counter)
for row in rows:
    versions_by_source_area[row['source_area']][row['version']] += 1

def choose_version(version_counts):
    if version_counts.get('latest', 0) > 0:
        return 'latest'
    if version_counts.get('stable', 0) > 0:
        return 'stable'
    return sorted(version_counts)[-1]

curated_prefixes = []
for source_area, counts in sorted(versions_by_source_area.items()):
    version = choose_version(counts)
    prefix = f'https://docs.example.com/platform/{source_area}/{version}'
    curated_prefixes.append(prefix)
    print(source_area, '->', version, dict(counts))

curated_prefixes


The output of curation is an explicit allowlist for your crawler. Linked binary and text file handling should be configured with narrow host allowlists.


In [ ]:
crawler_payload = {
    'start_url': 'https://docs.example.com/platform/',
    'max_depth': None,
    'extract_linked_files': True,
    'allowed_url_prefixes': curated_prefixes,
    'unblock_url_patterns': ['github.com/example-vendor'],
    'binary_host_allowlist': [
        'docs.example.com',
        'assets.example.com',
        'github.com/example-vendor/',
        'raw.githubusercontent.com/example-vendor/',
    ],
}

print(json.dumps(crawler_payload, indent=2))


To run against a real sitemap, use the CLI. Keep it as an inspected command first, then run it from a terminal or guarded notebook cell.


In [ ]:
cmd = [
    sys.executable,
    'scripts/sitemap_to_inventory.py',
    '--sitemap', 'https://docs.example.com/platform/sitemap.xml',
    '--path-prefix', '/platform/',
    '--output', 'tutorial/_outputs/inventory.csv',
]
print(' '.join(cmd))
